[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S09/2_Practica/en_clase/S09_01_clustering_en_clase.ipynb)

# S09 · 01 — Segmentar huéspedes con clustering · versión en clase

**Tiempo estimado:** 45 min en clase · **Sprint 9** · M3 (UD 3.9, clustering)

📖 *¿El código se hace largo? Léelo [explicado por escrito](https://github.com/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S09/2_Practica/codigo_explicado/S09_01_clustering.md): recorrido, glosario y qué hace cada celda.* <!--codigo-explicado-->

**Contexto TurisData.** TurisData Canarias tiene 1 200 huéspedes y hasta ahora
les manda a todos la misma newsletter. Dirección pregunta: *¿hay grupos de clientes
parecidos entre sí, y merece la pena tratarlos de forma distinta?*
Aquí **no hay etiquetas**: nadie nos ha dicho a qué grupo pertenece cada huésped.
Es aprendizaje **no supervisado**: dejamos que los datos se agrupen solos y luego
**nosotros** interpretamos qué significa cada grupo.

**Al terminar sabrás:**
1. Por qué hay que escalar las variables antes de agrupar.
2. Aplicar k-means y elegir el número de grupos con el coeficiente de silueta.
3. Leer un dendrograma y aplicar clustering jerárquico.
4. Usar DBSCAN y entender qué es el "ruido".
5. Poner nombre e interpretar cada segmento para el negocio.

> 🏫 **Versión «en clase»** · ⏱ ≈ 45 min <!--reparto-->
>
> - Es **la parte de este cuaderno que se hace en el aula**, con el docente: ejercicios **1, 2**.
> - Lo demás (ejercicios 🏠 y mini-reto) está en el [cuaderno completo](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S09/2_Practica/para_casa/S09_01_clustering.ipynb), para hacer en casa.


In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import linkage, dendrogram

SEMILLA = 42
np.random.seed(SEMILLA)
sns.set_theme(style="whitegrid")

huespedes = pd.read_csv(dato("huespedes.csv"))
print(huespedes.shape)
huespedes.head()

Cada fila es un huésped con 7 variables de comportamiento (más su identificador).
Fíjate en las **escalas**: la edad va de 18 a 75, el gasto diario de 35 a 340 y
`viaja_con_ninos` solo vale 0 ó 1.

In [ ]:
huespedes.describe().round(1).T

## 1. Escalar: que todas las variables "pesen" lo mismo

**Idea.** Clustering agrupa por **distancia**. Si una variable tiene números
grandes (gasto en euros) y otra pequeños (0/1 de niños), la grande manda y la
pequeña no cuenta. Solución: **estandarizar** (restar la media y dividir por la
desviación típica) para que todas tengan media 0 y desviación 1.

**Ejemplo resuelto** con una sola columna:

In [ ]:
edad = huespedes["edad"]
edad_escalada = (edad - edad.mean()) / edad.std()
print("media:", round(edad_escalada.mean(), 3), "· desviación:", round(edad_escalada.std(), 3))

### Ejercicio 1 · 🏫 en clase
Crea la matriz `X` con **todas las variables menos `id_huesped`**, estandarizadas con
`StandardScaler` (usa `fit_transform`). `X` debe ser una matriz de 1200 filas y 7 columnas.
Guarda también los nombres de las variables en la lista `variables`. Ojo: más abajo (apartado 3) se añade a la tabla la
columna `grupo`; si repites esta celda después, **no la incluyas** en `variables` (o reinicia el cuaderno).

In [ ]:
# TODO: Quita la columna id_huesped con drop(columns=...) y pásale el resultado a StandardScaler().fit_transform
...

assert X.shape == (1200, 7), "X debe tener 1200 filas y 7 columnas (sin id_huesped ni grupo)"
assert abs(X.mean()) < 1e-6 and abs(X[:, 0].std() - 1) < 1e-6, "Revisa que estés escalando con StandardScaler"
print("Ejercicio 1 correcto ✔")

## 2. k-means: agrupar alrededor de centros

**Idea.** Le decimos cuántos grupos queremos (`k`). El algoritmo coloca `k` centros,
asigna cada huésped al centro más cercano, recalcula los centros y repite hasta que
no cambian. Es rápido y fácil de explicar. Su límite: **tú decides `k`** y prefiere
grupos "redondos" y de tamaño parecido.

**Ejemplo resuelto** con `k = 3`:

In [ ]:
km3 = KMeans(n_clusters=3, n_init=10, random_state=SEMILLA).fit(X)
print("Tamaño de cada grupo:", np.bincount(km3.labels_))
print("Inercia (suma de distancias al centro):", round(km3.inertia_))

### ¿Cuántos grupos? El coeficiente de silueta
Para cada huésped compara *lo cerca que está de su grupo* con *lo cerca que está del
grupo vecino*. Va de -1 a 1: **cuanto más alto, grupos más compactos y separados**.
Por encima de 0,5 se considera una buena estructura; entre 0,25 y 0,5, aceptable.

### Ejercicio 2 · 🏫 en clase
Calcula en el diccionario `siluetas` la silueta media (`silhouette_score(X, etiquetas)`)
de k-means para cada `k` de 2 a 8 (clave = `k`, valor = silueta). Después guarda en
`k_mejor` el `k` con silueta más alta. Usa `n_init=10` y `random_state=SEMILLA`.

In [ ]:
# TODO: Bucle for k in range(2, 9): entrena KMeans, saca .labels_ y calcula silhouette_score; luego max(siluetas, key=siluetas.get)
...

assert set(siluetas) == set(range(2, 9)), "siluetas debe tener las claves 2, 3, ..., 8"
assert 3 <= k_mejor <= 5, "Revisa: k_mejor debería ser el k con la silueta más alta (está entre 3 y 5)"
print("k con mejor silueta:", k_mejor)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(list(siluetas), list(siluetas.values()), marker="o")
ax.axvline(k_mejor, color="tab:red", linestyle="--", label=f"mejor k = {k_mejor}")
ax.set_xlabel("Número de grupos (k)")
ax.set_ylabel("Silueta media")
ax.set_title("¿Cuántos segmentos de huéspedes hay?")
ax.legend()
plt.show()

---

## 🏁 Fin de la parte de clase

- Lo que queda (los ejercicios 🏠 y el mini-reto) está en el **[cuaderno completo](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S09/2_Practica/para_casa/S09_01_clustering.ipynb)**.
- Ahí también tienes, en cada ejercicio de casa, su celda plegada «🆘 Solución de referencia» por si te quedas atascado.
